# 01 · WHO FluNet 中国 H3N2 数据探索

本 Notebook 用于：
1. 加载清洗后的 FluNet 中国 H3N2 周度阳性率数据
2. 可视化双峰季节性特征（冬峰 + 夏峰）
3. 验证谐波回归季节参数
4. 输出供论文使用的数据探索图

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

from plots._style import setup_style
setup_style()

DATA_DIR = Path('../data/processed')
FIG_DIR  = Path('../output/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. 加载数据

In [ ]:
weekly_path = DATA_DIR / 'weekly_positivity.csv'

if not weekly_path.exists():
    print('数据文件不存在，请先运行:')
    print('  python data/fetch_flunet.py')
else:
    df = pd.read_csv(weekly_path, parse_dates=['date'])
    print(f'数据形状: {df.shape}')
    print(f'时间范围: {df["date"].min().date()} → {df["date"].max().date()}')
    print(f'H3N2 阳性率统计:')
    print(df['h3n2_pos_rate'].describe().round(4))
    df.head()

## 2. 全时序 H3N2 阳性率

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('中国 H3N2 流感周阳性率（2015–2024）\nWHO FluNet 数据', fontsize=14, fontweight='bold')

# 上图：全时序
ax1 = axes[0]
ax1.plot(df['date'], df['h3n2_pos_rate'], color='#2196F3', linewidth=1.5, alpha=0.8)
ax1.fill_between(df['date'], df['h3n2_pos_rate'], alpha=0.15, color='#2196F3')

# 标注峰值年
for year in range(2015, 2025):
    mask = df['date'].dt.year == year
    if mask.sum() > 0:
        peak_idx = df.loc[mask, 'h3n2_pos_rate'].idxmax()
        ax1.annotate(
            f"{year}\n{df.loc[peak_idx,'h3n2_pos_rate']:.2f}",
            xy=(df.loc[peak_idx, 'date'], df.loc[peak_idx, 'h3n2_pos_rate']),
            xytext=(0, 8), textcoords='offset points',
            fontsize=7, ha='center', color='#F44336'
        )

ax1.set_ylabel('H3N2 周阳性率')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax1.set_title('全时序 H3N2 周阳性率')

# 下图：按 ISO 周号叠加各年（展示季节性）
ax2 = axes[1]
df['iso_week_num'] = df['date'].dt.isocalendar().week
colors_yr = plt.cm.Blues(np.linspace(0.3, 1.0, 10))

for i, year in enumerate(range(2015, 2025)):
    yr_data = df[df['date'].dt.year == year].copy()
    if len(yr_data) > 0:
        ax2.plot(yr_data['iso_week_num'], yr_data['h3n2_pos_rate'],
                 color=colors_yr[i], linewidth=1.5, label=str(year), alpha=0.8)

ax2.set_xlabel('ISO 周号')
ax2.set_ylabel('H3N2 周阳性率')
ax2.set_title('各年 H3N2 阳性率季节模式（叠加）')
ax2.legend(loc='upper right', fontsize=8, ncol=2)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout()
plt.savefig(FIG_DIR / '01_flunet_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()
print('图表已保存')

## 3. 双峰特征分析

In [ ]:
# 按峰值标签分析
peak_stats = df.groupby('peak_label')['h3n2_pos_rate'].agg(['mean','max','count'])
peak_stats.columns = ['平均阳性率', '最大阳性率', '周数']
print('双峰季节统计:')
print(peak_stats.round(4))

# 年度峰值汇总
summary_path = DATA_DIR / 'seasonal_summary.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    print('\n年度峰值汇总:')
    print(summary.to_string(index=False))

## 4. 谐波回归季节参数验证

In [ ]:
# 加载季节参数
sp_path = DATA_DIR / 'seasonal_params.json'
if sp_path.exists():
    with open(sp_path) as f:
        sp = json.load(f)
    print('谐波回归季节参数:')
    for k, v in sp.items():
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

    # 绘制拟合曲线
    from model.params import ModelParams
    from model.seasonal import beta_series
    
    p = ModelParams(
        delta1=sp['delta1'], delta2=sp['delta2'],
        phi1=sp['phi1_rad'], phi2=sp['phi2_rad']
    )
    t_arr = np.arange(0, 365)
    b_arr = beta_series(t_arr, p) / p.beta0  # 归一化

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t_arr, b_arr, color='#7C4DFF', linewidth=2, label='β(t)/β₀ 谐波拟合')
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('一年中的天数 (DOY)')
    ax.set_ylabel('β(t)/β₀ 归一化系数')
    ax.set_title(f'双谐波季节函数 (δ₁={sp["delta1"]:.3f}, δ₂={sp["delta2"]:.3f}, R²={sp["fit_r2"]:.3f})')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01_harmonic_fit.png', dpi=300, bbox_inches='tight')
    plt.show()